# 11 — Who attends to whom? Decoder attention at mask ratio 0.5

**Question.** When a station is hidden from the encoder, the decoder has to
reconstruct it from *other* stations. Which ones does it actually read? Are
they the nearby ones, the ones at similar altitude, or does attention spread
out indistinguishably? And do v27 (Huber) and v30-nll (Gaussian NLL) solve it
the same way?

**Where the information lives.** A masked station contributes **no encoder
token at all** — masking removes whole stations for the whole window. So the
only routes into a masked station's prediction are:

1. `cross_attn` — its query reads the *visible* stations' encoder tokens.
2. `self_attn` — its query reads other queries, including masked ones.

`cross_attn` is the one that carries observations, so it is the focus here.

**Tensor layouts** (read from `src/model/decoder.py` / `encoder.py`, not assumed):

| tensor | shape | index arithmetic |
|---|---|---|
| decoder queries | `(B, N, K, d) -> (B, N*K, d)` | `q = n*K + k` — station-major, lead fastest |
| encoder context | `(B, T, N_vis, d) -> (B, T*N_vis, d)` | `j = t*N_vis + nv` — time-major, station fastest |
| cross-attn weights | `(B, heads, N*K, T*N_vis)` | rows sum to 1 over the kv axis |

Summing the kv axis over `T` collapses "which timestep" and leaves **share of
attention given to each visible station**, which is what we want. Both
reshapes are verified numerically in section 2 before any of it is
interpreted — including a causal check that perturbing a station actually
moves the predictions that attend to it.

**Note on masks.** This notebook draws its own mask (seeded), so the hidden set
will NOT match the one in `predictions.pt` — reproducing that needs the dump's
exact batch size and window order. Nothing here depends on matching it; the
masked set actually drawn is recorded and used throughout.

In [ ]:
# ── Bootstrap ────────────────────────────────────────────────────────────────
import os, sys, math, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
for _c in (os.getcwd(), os.path.join(os.getcwd(), "notebooks", "analysis")):
    if os.path.isfile(os.path.join(_c, "common.py")):
        if _c not in sys.path: sys.path.insert(0, _c)
        break
import importlib
import common as C
# common.py is edited while these notebooks are open. A plain import is cached
# by the kernel, so edits are invisible until a restart — and a stale module
# silently serves stale caches (this is exactly how a station_table without
# easting/northing survived a fix to station_table). Reload every run.
importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

MODELS_TO_PROBE = ["v27", "v30-nll"]      # both were trained at mask_ratio 0.5
MASK_RATIO      = 0.5
N_WINDOWS       = 24        # windows to average attention over (B=1 each)
SEED            = 42
torch.manual_seed(SEED); np.random.seed(SEED)

stn = C.station_table(); ns = C.norm_stats()
VARS = ns["var_names"]; N = len(stn)
XY = stn[["longitude", "latitude"]].values
E, Nn = stn["easting"].values, stn["northing"].values
DIST = np.sqrt((E[:, None]-E[None, :])**2 + (Nn[:, None]-Nn[None, :])**2)/1000.0
print(f"{N} stations · probing {MODELS_TO_PROBE} at mask_ratio {MASK_RATIO}")

## 1. Rebuild the models and capture attention

`nn.MultiheadAttention` is called with `need_weights=False` so PyTorch can use
the fused kernel, which never materialises the weight matrix. We wrap the two
attention modules of each decoder block to force `need_weights=True` and
`average_attn_weights=False` (per-head), then **reduce inside the wrapper** —
the raw cross-attention tensor is `(1, 8, N*K, T*N_vis)` ≈ 120 MB per block,
while the reduced form is under a megabyte.

In [ ]:
# ── Dataset + model rebuild (mirrors test.py) ───────────────────────────────
from data.dataset import load_peakweather, StationMAEDataset
from model.mae import StationMAE

ds = load_peakweather(root=C.data_root())
def ckpt_path(run):
    p = os.path.join(C.PROJ, "checkpoints", f"full_run_cloud_{run}", "best.ckpt")
    assert os.path.isfile(p), p
    return p

def build(run):
    ck = torch.load(ckpt_path(run), map_location="cpu", weights_only=False)
    cfg = ck["hyper_parameters"]["cfg"]
    m = StationMAE.from_cfg(cfg, dropout=0.0, drop_path_rate=0.0)
    sd = {}
    for k, v in ck["state_dict"].items():
        if not k.startswith("model."): continue
        k = k[len("model."):]
        if k.startswith("_orig_mod."): k = k[len("_orig_mod."):]
        sd[k] = v
    missing, unexpected = m.load_state_dict(sd, strict=False)
    assert not [x for x in list(missing)+list(unexpected) if "blocks" in x], \
        f"structural mismatch for {run}: {missing[:5]} {unexpected[:5]}"
    m.eval(); m.encoder.mask_ratio = MASK_RATIO
    return m, cfg

MODELS = {}
for r in MODELS_TO_PROBE:
    MODELS[r], _cfg = build(r)
    print(f"  {r:10s} rebuilt · dec_layers={_cfg['dec_layers']} "
          f"heads={_cfg['dec_heads']} d_model={_cfg['d_model']} "
          f"trained mask_ratio={_cfg['mask_ratio']}")

# Mirrors src/test.py exactly. Getting any of these wrong changes the task:
#   delta_mode="fixed_grid" + num_delta_per_sample=1 is what produces the
#   K-lead grid (test.py:440-449); obs_stats must come from the TRAIN split or
#   the normalisation differs from the dumps; the arg is train_stride, not
#   stride, and it is what makes the window set match.
_cfgd = _cfg
cache_dir = C.data_root()
train_ds = StationMAEDataset(
    ds, window_size=_cfgd["window"], delta_steps=_cfgd["max_delta"],
    split="train", num_delta_per_sample=1,
    max_delta_steps=_cfgd["max_delta"], cache_dir=cache_dir,
    exclude_stations=C.EXCLUDE, delta_mode="fixed_grid",
    delta_grid_stride=_cfgd["delta_grid_stride"])
obs_stats = train_ds.obs_stats

test_ds = StationMAEDataset(
    ds, window_size=_cfgd["window"], delta_steps=_cfgd["max_delta"],
    split="test", obs_stats=obs_stats, num_delta_per_sample=1,
    max_delta_steps=_cfgd["max_delta"], cache_dir=cache_dir,
    exclude_stations=C.EXCLUDE, delta_mode="fixed_grid",
    delta_grid_stride=_cfgd["delta_grid_stride"],
    index_mode="sliding", train_stride=9)
K_LEADS = len(test_ds.delta_grid)
T_PATCH = _cfgd["window"] // _cfgd["temporal_patch"]
print(f"  test windows: {len(test_ds):,}   K={K_LEADS} leads   "
      f"T={T_PATCH} patch tokens")

In [ ]:
# ── Attention capture ───────────────────────────────────────────────────────
# The reduction happens INSIDE the wrapper. Raw cross-attention is
# (1, 8, N*K, T*N_vis) ~= 120 MB per block and self-attention (1, 8, N*K, N*K)
# ~= 130 MB; reduced they are under a megabyte, which is what makes averaging
# over many windows possible at all.
def attach(model, store, N, T):
    undo = []
    for bi, blk in enumerate(model.decoder.blocks):
        for name in ("cross_attn", "self_attn"):
            mod = getattr(blk, name)
            orig = mod.forward
            def make(orig=orig, bi=bi, name=name):
                def fwd(q, k, v, **kw):
                    kw = dict(kw)
                    kw["need_weights"] = True
                    kw["average_attn_weights"] = False
                    out, w = orig(q, k, v, **kw)
                    w = w.detach().float()[0]              # (heads, Lq, Lkv)
                    H, Lq, Lkv = w.shape
                    K = Lq // N
                    if name == "cross_attn":
                        Nv = Lkv // T
                        r = w.reshape(H, N, K, T, Nv).sum(3).mean(0)  # (N,K,Nv)
                    else:
                        # queries only: collapse both lead axes -> (N, N)
                        r = w.reshape(H, N, K, N, K).mean((0, 2, 4))
                    store.setdefault(name, []).append(r.cpu().numpy())
                    return out, w
                return fwd
            mod.forward = make()
            undo.append(mod)
    return undo

def detach_all(undo):
    # delete the instance attribute so the class method is used again; simply
    # reassigning would leave a bound-method attribute shadowing it forever
    for mod in undo:
        try: del mod.forward
        except AttributeError: pass

@torch.no_grad()
def capture(model, idx, seed=None):
    """One window -> (cross (N,K,N_vis), self (N,N), masked_idx, vis, preds, batch).

    seed is applied immediately before the forward: _mask_stations draws from
    the global RNG on EVERY pass, so two forwards without re-seeding see two
    different masked sets.
    """
    b = test_ds[idx]
    batch = {k: (v.unsqueeze(0) if torch.is_tensor(v) else v) for k, v in b.items()}
    store = {}
    undo = attach(model, store, N, T_PATCH)
    try:
        if seed is not None:
            torch.manual_seed(seed)
        out = model.forward_multi_delta(
            batch["x"], batch["x_mask"], batch["spatial"][0], batch["x_hours"],
            batch["y"], batch["y_mask"], batch["y_hours"], batch["delta_steps"])
    finally:
        detach_all(undo)
    preds, masked_idx = out[1], out[2]
    mi = masked_idx[0].numpy(); vis = np.setdiff1d(np.arange(N), mi)
    cross = np.mean(store["cross_attn"], axis=0)     # mean over decoder blocks
    selfa = np.mean(store["self_attn"],  axis=0)
    assert cross.shape[2] == len(vis), (cross.shape, len(vis))
    return cross, selfa, mi, vis, preds[0].numpy(), batch

_c, _s, _mi, _vis, _pr, _b = capture(MODELS["v27"], 0, seed=SEED)
print(f"cross {_c.shape} (station, lead, visible)   self {_s.shape}")
print(f"masked {_mi.shape[0]} of {N}   "
      f"rows sum to {_c[0,0].sum():.4f} (must be 1.0)")

## 2. Verify the layout before believing anything

Two checks. The cheap one: attention rows must sum to 1 after collapsing the
time axis. The decisive one is **causal** — perturb a single visible station's
inputs and measure how much each masked station's prediction moves. If the
reshape is right, the stations that attend most to the perturbed one must be
the ones that move most. A transposed or mis-strided reshape would destroy
that correlation while leaving the row sums untouched.

In [ ]:
@torch.no_grad()
def perturb_test(model, idx=0, donor_rank=0, delta=3.0, seed=SEED):
    """Causal check: bump ONE visible station, see whose predictions move.

    Both forwards are seeded identically. Without that, _mask_stations redraws
    and the second pass masks a different set of stations — the difference
    would then be dominated by the mask change, not by the perturbation, and
    the check would look like noise no matter how correct the reshape is.
    """
    cross, _, mi, vis, base, batch = capture(model, idx, seed=seed)
    att_from_masked = cross[mi].mean(axis=(0, 1))          # (N_vis,)
    donor_local = int(np.argsort(att_from_masked)[::-1][donor_rank])
    donor = int(vis[donor_local])

    x2 = batch["x"].clone()
    x2[:, :, donor, 0] += delta                            # temperature bump
    torch.manual_seed(seed)                                # identical mask
    out2 = model.forward_multi_delta(
        x2, batch["x_mask"], batch["spatial"][0], batch["x_hours"],
        batch["y"], batch["y_mask"], batch["y_hours"], batch["delta_steps"])
    mi2 = out2[2][0].numpy()
    assert np.array_equal(np.sort(mi), np.sort(mi2)), \
        "mask changed between the two passes — the seed did not take"
    moved = np.abs(out2[1][0].numpy() - base)              # (K, N, V)
    return donor, donor_local, cross, mi, vis, moved

donor, dloc, cross, mi, vis, moved = perturb_test(MODELS["v27"])
print(f"perturbed visible station {stn.abbr[donor]} (index {donor}) by +3 °C\n")
att    = cross[mi][:, :, dloc].mean(1)                     # (n_masked,)
delta_ = moved[:, mi, 0].mean(0)                           # (n_masked,)
ok = np.isfinite(att) & np.isfinite(delta_)
r = np.corrcoef(np.argsort(np.argsort(att[ok])),
                np.argsort(np.argsort(delta_[ok])))[0, 1]
print(f"Spearman(attention to donor, |change in prediction|) = {r:+.3f}")
print("  what to expect: a WRONG station reshape scrambles the mapping and")
print("  drives this to ~0, so any clearly non-zero value confirms the layout.")
print("  Do not expect it near 1: attention weight is not attribution — a large")
print("  weight on a low-variance token moves the output less than a small")
print("  weight on an informative one. rho ~ 0.4 (rho^2 ~ 0.2) is a pass.")
print("  The decisive test is the permutation null in the next cell.")
for i_ in np.argsort(att)[::-1][:5]:
    print(f"    {stn.abbr[mi[i_]]:5s} att={att[i_]:.4f}  moved={delta_[i_]:.4f} °C  "
          f"dist={DIST[mi[i_], donor]:5.1f} km")

In [ ]:
# ── Hardened verification: permutation null, several donors and windows ─────
# One donor in one window is a single experiment. Three additions:
#   1. PERMUTATION NULL — shuffle attention across masked stations and redo the
#      correlation. This is the decisive control: it destroys the
#      station->attention mapping while leaving both marginal distributions
#      untouched, so the null says exactly what "no mapping" looks like.
#   2. Repetition over donors and windows, for a distribution not a point.
#   3. PARTIAL correlation controlling distance — attention and sensitivity
#      both decay with separation, so part of the raw rho could be geometry
#      rather than routing. What survives is attention's own contribution.
def _rank(x): return np.argsort(np.argsort(x)).astype(float)
def _sp(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 8: return np.nan
    return float(np.corrcoef(_rank(a[m]), _rank(b[m]))[0, 1])
def _partial(a, b, c):
    """Spearman(a,b) with c partialled out."""
    rab, rac, rbc = _sp(a, b), _sp(a, c), _sp(b, c)
    den = np.sqrt(max(1 - rac**2, 1e-12) * max(1 - rbc**2, 1e-12))
    return (rab - rac * rbc) / den

rng = np.random.default_rng(0)
step = max(1, len(test_ds) // 4)
rows, nulls = [], []
for wi in range(3):
    for dr in range(4):
        dn, dl, cr, mi_, vs_, mv = perturb_test(
            MODELS["v27"], idx=wi * step, donor_rank=dr, seed=SEED + wi)
        att = cr[mi_][:, :, dl].mean(1)
        dmv = mv[:, mi_, 0].mean(0)
        dst = DIST[mi_, dn]
        rows.append({"window": wi, "donor": stn.abbr[dn], "rank": dr,
                     "rho": _sp(att, dmv),
                     "rho_partial_dist": _partial(att, dmv, dst),
                     "rho_dist_only": _sp(-dst, dmv),
                     "top1_share": att.max() / max(att.sum(), 1e-12)})
        nulls += [_sp(rng.permutation(att), dmv) for _ in range(100)]

V = pd.DataFrame(rows)
nulls = np.array(nulls)
display(V.round(3))
print(f"\nTRUE   rho: mean {V.rho.mean():+.3f}  min {V.rho.min():+.3f}  "
      f"max {V.rho.max():+.3f}   (n={len(V)} donor x window trials)")
print(f"NULL   rho: mean {nulls.mean():+.3f}  95th pct {np.percentile(nulls,95):+.3f}  "
      f"max {nulls.max():+.3f}   ({len(nulls):,} permutations)")
frac = (nulls >= V.rho.mean()).mean()
print(f"       permutations reaching the observed mean: {frac*100:.2f}%")
print(f"\nPARTIAL rho (distance controlled): mean "
      f"{V.rho_partial_dist.mean():+.3f}")
print(f"DISTANCE alone  (-dist vs moved) : mean {V.rho_dist_only.mean():+.3f}")
print("\nVERDICT")
if V.rho.mean() > np.percentile(nulls, 99) and V.rho.mean() > 0.15:
    print("  PASS — the true correlation sits far outside the permutation null,")
    print("  so the station->attention mapping carries real information.")
    if V.rho_partial_dist.mean() > 0.10:
        print("  Attention also survives controlling for distance: it is not a")
        print("  pure geometry proxy, it routes to specific stations.")
    else:
        print("  NOTE: little survives once distance is controlled — attention")
        print("  here may be largely a distance prior. Treat section 4's donor")
        print("  identities cautiously.")
else:
    print("  FAIL — indistinguishable from the permuted null. STOP: the kv or")
    print("  query reshape is wrong and nothing below is interpretable.")

## 3. Does attention follow distance?

For every masked station, the share of attention it gives to each visible
station, binned by separation. A model exploiting spatial structure should
show a clear decay; a flat curve would mean it is averaging the network.

In [ ]:
def collect(model, n_windows=N_WINDOWS):
    """Cross-attention for several windows, with per-window masks recorded.

    Window j is seeded with SEED + j, so EVERY model sees the identical masked
    set for a given window. That is what makes the v27 vs v30-nll comparison in
    section 6 paired rather than a comparison of two different problems.
    """
    W = []
    step = max(1, len(test_ds)//n_windows)
    for j in range(n_windows):
        cr, sf, mi, vis, _, _ = capture(model, j*step, seed=SEED + j)
        W.append({"cross": cr, "self": sf, "mi": mi, "vis": vis})
    return W

ATT = {r: collect(MODELS[r]) for r in MODELS_TO_PROBE}
print({r: len(v) for r, v in ATT.items()})

EDGES = np.array([0, 5, 10, 20, 30, 50, 75, 100, 150, 250])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for r in MODELS_TO_PROBE:
    d_all, a_all = [], []
    for w in ATT[r]:
        for qi, s in enumerate(w["mi"]):                   # masked queries only
            a_all.append(w["cross"][s].mean(0))            # (N_vis,) over leads
            d_all.append(DIST[s, w["vis"]])
    d_all = np.concatenate(d_all); a_all = np.concatenate(a_all)
    bi = np.digitize(d_all, EDGES[1:-1])
    y = [a_all[bi == b].mean() for b in range(len(EDGES)-1)]
    axes[0].plot(EDGES[:-1], y, "o-", ms=4, color=C.MODELS[r][1],
                 label=C.MODELS[r][0])
    # uniform reference: 1/N_vis
    axes[1].plot(EDGES[:-1], np.array(y)*len(ATT[r][0]["vis"]), "o-", ms=4,
                 color=C.MODELS[r][1], label=C.MODELS[r][0])
axes[0].set_ylabel("mean attention weight"); axes[0].set_yscale("log")
axes[1].axhline(1.0, color="k", lw=1, ls=":")
axes[1].set_ylabel("attention / uniform (1 = indifferent)")
for ax in axes:
    ax.set_xlabel("distance from masked station to visible station [km]")
    ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.suptitle("Attention paid by MASKED stations, by separation", y=1.02)
plt.tight_layout(); C.save_fig(fig, "11_attention_vs_distance"); plt.show()

## 4. Who are the donors? Concentration, and a map

In [ ]:
rows = []
for r in MODELS_TO_PROBE:
    for w in ATT[r]:
        for s in w["mi"]:
            a = w["cross"][s].mean(0)                      # (N_vis,)
            o = np.argsort(a)[::-1]
            p = a/np.maximum(a.sum(), 1e-12)
            rows.append({
                "model": r, "station": stn.abbr[s], "s": s,
                "top1_share": p[o[0]], "top5_share": p[o[:5]].sum(),
                "entropy": float(-(p*np.log(np.maximum(p, 1e-12))).sum()),
                "eff_donors": float(np.exp(-(p*np.log(np.maximum(p,1e-12))).sum())),
                "mean_dist": float((p*DIST[s, w["vis"]]).sum()),
                "top1_dist": float(DIST[s, w["vis"][o[0]]]),
                "top1_dz": float(abs(stn.height[s]-stn.height[w["vis"][o[0]]])),
                "height": stn.height[s]})
ATTDF = pd.DataFrame(rows)
summ = ATTDF.groupby("model")[["top1_share","top5_share","eff_donors",
                               "mean_dist","top1_dist","top1_dz"]].mean()
summ["uniform_eff_donors"] = len(ATT[MODELS_TO_PROBE[0]][0]["vis"])
display(summ.round(3).style.set_caption(
    "Averaged over masked stations and windows. eff_donors = exp(entropy): "
    "how many visible stations the query effectively reads. Compare with "
    "uniform_eff_donors — the value if attention were indifferent."))
C.save_table(ATTDF, "11_attention_per_masked_station")

per = ATTDF.groupby(["model","station"]).agg(
    eff_donors=("eff_donors","mean"), mean_dist=("mean_dist","mean"),
    height=("height","first")).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for r in MODELS_TO_PROBE:
    q = per[per.model == r]
    axes[0].scatter(q.height, q.eff_donors, s=14, alpha=.6,
                    color=C.MODELS[r][1], label=C.MODELS[r][0])
    axes[1].scatter(q.height, q.mean_dist, s=14, alpha=.6,
                    color=C.MODELS[r][1], label=C.MODELS[r][0])
axes[0].set_ylabel("effective number of donors"); axes[1].set_ylabel("mean donor distance [km]")
for ax in axes: ax.set_xlabel("masked station height [m]"); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.suptitle("Do isolated / high stations gather from further away?", y=1.02)
plt.tight_layout(); C.save_fig(fig, "11_donor_concentration"); plt.show()

In [ ]:
# ── Map: top-3 donors for a few masked stations (v27, first window) ─────────
w = ATT["v27"][0]
pick = list(w["mi"][:6])
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.scatter(XY[:,0], XY[:,1], s=8, c="#CCC", zorder=1, label="visible")
ax.scatter(XY[w["mi"],0], XY[w["mi"],1], s=26, c="#C4502A", marker="x",
           zorder=3, label="masked")
for s in pick:
    a = w["cross"][s].mean(0); o = np.argsort(a)[::-1][:3]
    for rank, j in enumerate(o):
        d = w["vis"][j]
        ax.annotate("", xy=(XY[d,0], XY[d,1]), xytext=(XY[s,0], XY[s,1]),
                    arrowprops=dict(arrowstyle="->", color="#1F5F6B",
                                    lw=2.2-0.6*rank, alpha=.8-0.2*rank), zorder=2)
    ax.annotate(stn.abbr[s], (XY[s,0], XY[s,1]), fontsize=7,
                xytext=(3,3), textcoords="offset points", zorder=4)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.legend(fontsize=8)
ax.set_title("v27: top-3 attended visible stations for six masked stations")
ax.grid(alpha=.3)
C.save_fig(fig, "11_donor_map"); plt.show()

## 5. Masked vs visible queries, and lead-time dependence

A *visible* station's query can read its own encoder tokens — the easy route.
A masked one cannot. If the model uses attention sensibly, masked queries
should be more diffuse (gathering from several neighbours) and visible queries
sharply self-focused. Lead time is the second axis: at +6h a station's own
recent history is worth less, so attention may broaden.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for r in MODELS_TO_PROBE:
    self_share, eff_m, eff_v = [], [], []
    for w in ATT[r]:
        loc = {s: i for i, s in enumerate(w["vis"])}
        for s in w["vis"]:                        # visible: self-attention share
            a = w["cross"][s].mean(0); p = a/np.maximum(a.sum(),1e-12)
            self_share.append(p[loc[s]])
            eff_v.append(np.exp(-(p*np.log(np.maximum(p,1e-12))).sum()))
        for s in w["mi"]:
            a = w["cross"][s].mean(0); p = a/np.maximum(a.sum(),1e-12)
            eff_m.append(np.exp(-(p*np.log(np.maximum(p,1e-12))).sum()))
    axes[0].hist(self_share, bins=40, alpha=.55, color=C.MODELS[r][1],
                 label=f"{C.MODELS[r][0]} (visible)")
    axes[1].hist(eff_m, bins=30, alpha=.55, color=C.MODELS[r][1],
                 label=f"{C.MODELS[r][0]} masked")
    axes[1].hist(eff_v, bins=30, alpha=.30, color=C.MODELS[r][1], histtype="step",
                 lw=2, label=f"{C.MODELS[r][0]} visible")
    # lead-time dependence
    eff_by_k = []
    Kn = ATT[r][0]["cross"].shape[1]
    for k in range(Kn):
        vals = []
        for w in ATT[r]:
            for s in w["mi"]:
                p = w["cross"][s][k]; p = p/np.maximum(p.sum(),1e-12)
                vals.append(np.exp(-(p*np.log(np.maximum(p,1e-12))).sum()))
        eff_by_k.append(np.mean(vals))
    axes[2].plot(range(Kn), eff_by_k, "o-", ms=3, color=C.MODELS[r][1],
                 label=C.MODELS[r][0])
axes[0].set_xlabel("share of attention a VISIBLE station gives itself")
axes[1].set_xlabel("effective number of donors")
axes[2].set_xlabel("lead index"); axes[2].set_ylabel("effective donors (masked)")
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=7)
fig.suptitle("Attention concentration: masked vs visible, and vs lead time", y=1.03)
plt.tight_layout(); C.save_fig(fig, "11_concentration_masked_vs_visible"); plt.show()

## 6. Do v27 and v30-nll attend the same way?

Same architecture, same training mask ratio, different loss. If the attention
maps agree closely, the loss changed the head but not the information routing;
if they diverge, the NLL objective reshaped which neighbours the model trusts.

In [ ]:
common_w = min(len(ATT[MODELS_TO_PROBE[0]]), len(ATT[MODELS_TO_PROBE[1]]))
rs = []
for wi in range(common_w):
    a, b = ATT[MODELS_TO_PROBE[0]][wi], ATT[MODELS_TO_PROBE[1]][wi]
    if not np.array_equal(a["mi"], b["mi"]):
        continue                       # masks differ -> not comparable
    for s in a["mi"]:
        x, y = a["cross"][s].mean(0), b["cross"][s].mean(0)
        rs.append(np.corrcoef(x, y)[0, 1])
if rs:
    print(f"per-masked-station attention correlation, v27 vs v30-nll: "
          f"median {np.median(rs):+.3f}  (n={len(rs)})")
else:
    print("masks drawn independently per model — pairing not possible in this "
          "run; compare the aggregate statistics below instead.")
display(summ[["eff_donors","mean_dist","top1_share"]].round(3).style
        .set_caption("aggregate comparison (paired or not, these are "
                     "distribution-level and robust to mask differences)"))

## Interpretation

**How to read.** Section 2 is the gate: if the attention-vs-perturbation
correlation is not strongly positive, the reshape is wrong and nothing below
is meaningful. Section 3 answers the headline question — a decay with distance
means the model learned spatial structure; the right-hand panel expresses it as
a multiple of uniform attention (1/N_vis), so "2" reads as *twice what an
indifferent model would give*.

**Effective donors** (`exp(entropy)`) is the interpretable concentration
measure: 3 means the query behaves as if reading ~3 stations, against
`uniform_eff_donors` ≈ 78 if it were indifferent.

**Expected patterns, to be confirmed or refuted:** visible stations should
place most of their attention on themselves; masked stations should spread
over a handful of near neighbours; and the effective-donor count should rise
with lead time as recent history loses value. If isolated or high-altitude
stations show larger `mean_dist`, that connects directly to the masking
penalty in notebook 07 and the v31-v32 gap in notebook 05 — three independent
routes to the same claim about spatial context.

**Caveats.** Attention weight is not attribution: a large weight on a
low-variance token can matter less than a small weight on an informative one,
which is why section 2 measures actual sensitivity rather than trusting the
weights alone. Weights are averaged over both decoder blocks and all heads;
individual heads may specialise, and per-head maps are one `mean(0)` away if
that becomes interesting.